<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/p301_Transcriptomics_DEG_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Transcriptomics: Taxane-Induced DEG / Volcano Analysis

**Project:** Precision Oncology Mechanisms of Pacific Yew Taxanes
**Section mirrored:** Report §1–2 (G2/M arrest biomarkers) and Figure 2 (DEG volcano plot)

**What this notebook actually does**

The uploaded report presents a volcano plot of taxane-induced differentially expressed genes (DEGs) with
specific fold-changes and p-values, but gives no underlying count matrix, cohort, or DE method — those
numbers cannot be reproduced or verified as written. This notebook gives you a **working, swappable
pipeline** instead:

1. A clearly-labeled **synthetic RNA-seq count matrix** (tumor vs. taxane-treated) built around the gene set
   named in the report, so the whole pipeline runs end-to-end out of the box.
2. A real differential-expression routine (Welch's t-test + log2FC + Benjamini–Hochberg FDR) — the standard
   approach when you don't have DESeq2/R available in the Colab Python runtime.
3. A **drop-in slot** (Section 2) to replace the synthetic matrix with your own TCGA / GEO / in-house counts —
   at that point the volcano plot and gene calls become real results, not illustrations.

> ⚠️ **Until you load real counts in Section 2, every number downstream is simulated for demonstration only —
> do not cite it.**


In [1]:
# --- Setup ---
!pip -q install statsmodels
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.dpi"] = 120


## 1. Gene panel

Genes referenced in the report's mechanism section and volcano plot (Figure 2), plus a few standard
apoptosis/cell-cycle housekeeping controls.

In [2]:
genes_up = ["CDKN1A", "TUBB1", "CASP3", "CDK1", "CCNB1", "PLK1", "MAD2L1", "EGFR", "E2F1"]
genes_down = ["BCL2L1", "TP53", "BRCA1", "BRCA2", "SPOP"]
genes_ns = ["AR", "PTEN", "GATA3", "AURKA", "KRAS", "GAPDH", "MAP4"]
all_genes = genes_up + genes_down + genes_ns
print(f"{len(all_genes)} genes: {all_genes}")


21 genes: ['CDKN1A', 'TUBB1', 'CASP3', 'CDK1', 'CCNB1', 'PLK1', 'MAD2L1', 'EGFR', 'E2F1', 'BCL2L1', 'TP53', 'BRCA1', 'BRCA2', 'SPOP', 'AR', 'PTEN', 'GATA3', 'AURKA', 'KRAS', 'GAPDH', 'MAP4']


## 2. Load expression data

**Option A (default): synthetic demo matrix** — illustrates the workflow only.

**Option B (recommended for real analysis):** replace this cell with your own count matrix, e.g.:

```python
counts = pd.read_csv("/content/your_counts.csv", index_col=0)   # genes x samples
metadata = pd.read_csv("/content/your_metadata.csv", index_col=0)  # sample -> condition (control/treated)
```

or pull directly from TCGA/GDC via `TCGAbiolinks` (R) exported to CSV, or `pyGEO`/`GEOparse` for a GEO
series matched to taxane-treated cell lines (e.g. GSE datasets for paclitaxel-treated breast/ovarian lines).

In [ ]:
# ---- Option A: synthetic demo matrix (REPLACE with real data per markdown above) ----
n_control, n_treated = 12, 12
samples = [f"Ctrl_{i+1}" for i in range(n_control)] + [f"Taxane_{i+1}" for i in range(n_treated)]
condition = ["control"] * n_control + ["treated"] * n_treated

base_expr = np.random.normal(8, 1.2, size=len(all_genes))
counts = pd.DataFrame(index=all_genes, columns=samples, dtype=float)

effect = {g: np.random.uniform(1.5, 3.0) for g in genes_up}
effect.update({g: -np.random.uniform(1.2, 2.2) for g in genes_down})
effect.update({g: np.random.uniform(-0.15, 0.15) for g in genes_ns})

for i, gene in enumerate(all_genes):
    ctrl_vals = np.random.normal(base_expr[i], 0.6, n_control)
    treat_vals = np.random.normal(base_expr[i] + effect[gene], 0.6, n_treated)
    counts.loc[gene] = np.concatenate([ctrl_vals, treat_vals])

metadata = pd.DataFrame({"condition": condition}, index=samples)
counts.head()


## 3. Differential expression (log2FC + Welch's t-test + BH-FDR)

This is the same statistical logic DESeq2/limma implement, simplified to run natively in the Colab Python
kernel without R. For a manuscript, prefer running actual DESeq2 in R (or `pydeseq2`) on raw counts.

In [ ]:
ctrl_cols = metadata.index[metadata.condition == "control"]
treat_cols = metadata.index[metadata.condition == "treated"]

results = []
for gene in counts.index:
    c = counts.loc[gene, ctrl_cols].astype(float)
    t = counts.loc[gene, treat_cols].astype(float)
    log2fc = t.mean() - c.mean()  # values are already log-scale in the synthetic matrix
    tstat, pval = stats.ttest_ind(t, c, equal_var=False)
    results.append({"gene": gene, "log2FC": log2fc, "pvalue": pval})

deg = pd.DataFrame(results)
deg["padj"] = multipletests(deg["pvalue"], method="fdr_bh")[1]
deg["neglog10p"] = -np.log10(deg["padj"].clip(lower=1e-300))

FC_THRESH, P_THRESH = 1.5, 0.05
def call(row):
    if row.padj < P_THRESH and row.log2FC >= FC_THRESH:
        return "Upregulated (sensitizing)"
    if row.padj < P_THRESH and row.log2FC <= -FC_THRESH:
        return "Downregulated (sensitizing)"
    return "Not significant"

deg["call"] = deg.apply(call, axis=1)
deg = deg.sort_values("padj")
deg


## 4. Volcano plot (reproduces the layout of the report's Figure 2, from real computed statistics)

In [ ]:
colors = {"Upregulated (sensitizing)": "#2e7d32",
          "Downregulated (sensitizing)": "#c62828",
          "Not significant": "#9e9e9e"}

fig, ax = plt.subplots(figsize=(8, 6))
for call_type, sub in deg.groupby("call"):
    ax.scatter(sub.log2FC, sub.neglog10p, c=colors[call_type], label=call_type,
               edgecolor="black", linewidth=0.4, s=60, zorder=3)

for _, r in deg.iterrows():
    ax.annotate(r.gene, (r.log2FC, r.neglog10p), fontsize=8, xytext=(4, 3),
                textcoords="offset points")

ax.axvline(FC_THRESH, ls="--", c="gray", lw=0.8)
ax.axvline(-FC_THRESH, ls="--", c="gray", lw=0.8)
ax.axhline(-np.log10(P_THRESH), ls="--", c="gray", lw=0.8)
ax.set_xlabel("log2 Fold Change")
ax.set_ylabel("-log10 adjusted p-value")
ax.set_title("Transcriptomics Volcano Plot (Taxane-Induced DEGs in G2/M Arrest)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("figure2_deg_volcano.png", dpi=300)
plt.show()


## 5. Export

Saves the DEG table and figure for downstream use in Notebook 05 (manuscript dashboard).

In [ ]:
deg.to_csv("deg_results.csv", index=False)
print("Saved: deg_results.csv, figure2_deg_volcano.png")


---
### Next steps for a real manuscript
- Swap in TCGA-BRCA/OV/LUAD/PRAD RNA-seq (tumor vs. matched normal, or pre/post-taxane if a longitudinal
  cohort is available) via `TCGAbiolinks` or GDC API.
- Run actual DESeq2 (R) or `pydeseq2` on raw counts rather than the t-test approximation used here.
- Validate the DEG calls against an independent GEO series before drawing mechanistic conclusions.
